In [95]:
import pandas as pd
import numpy as np

# Giả sử 'df_raw' là dataframe chứa 31 thuộc tính ban đầu của bạn
# Load dataset

DATA_PATH = "../../data/final/scored_data.csv"

df_ready_for_kbrs = pd.read_csv(DATA_PATH)

In [96]:
filter_features = ['product_name', 'price_vnd', 'price_segment', 'weight_tier', 'score_perf', 'score_cam', 'score_batt', 'score_disp','rear_main_camera_mp', 'front_camera_mp','ip_status']
df_ready_for_kbrs[filter_features].describe().T.round(2)

,count,mean,std,min,25%,50%,75%,max
price_vnd,387.0,14264496.12,11821634.24,1590000.00,4940000.00,8990000.00,22940000.00,63990000.00
weight_tier,387.0,2.02,0.64,1.00,2.00,2.00,2.00,3.00
score_perf,387.0,4.18,2.01,0.48,2.63,3.64,5.82,8.99
score_cam,387.0,6.03,2.25,0.00,5.04,6.38,7.72,10.00
score_batt,387.0,4.21,1.40,0.91,3.16,3.98,5.01,8.00
score_disp,387.0,4.39,2.27,0.00,2.29,4.29,6.29,8.29
rear_main_camera_mp,380.0,56.21,42.30,8.00,48.00,50.00,50.00,200.00
front_camera_mp,384.0,16.43,10.63,5.00,10.00,12.00,18.00,50.00
ip_status,387.0,0.70,0.46,0.00,0.00,1.00,1.00,1.00


In [97]:
PERF_THRESHOLD = 5.8 #Lấy mức 75%
CAM_THRESHOLD = 7.7

MAPPING_MATRIX_CONFIG = {
    'Gia_Re': {
        'weights': {'perf': 0.25, 'cam': 0.25, 'batt': 0.25, 'disp': 0.25},
        'hard_filters': {'price_segment': lambda x: x == 'budget'}
    },
    'Flagship': {
        'weights': {'perf': 0.3, 'cam': 0.3, 'batt': 0.1, 'disp': 0.3},
        'hard_filters': {'price_segment': lambda x: x == 'flagship'}
    },
    'Lien_Lac_Co_Ban': {
        'weights': {'perf': 0.1, 'cam': 0.1, 'batt': 0.7, 'disp': 0.1},
        'hard_filters': {} # Không có điều kiện cứng
    },
    'Choi_Game': {
        'weights': {'perf': 0.7, 'cam': 0.05, 'batt': 0.15, 'disp': 0.1},
        'hard_filters': {
            'score_perf': lambda x: x >= PERF_THRESHOLD,      
            'refresh_rate_hz': lambda x: x >= 120.0, 
            'ram_gb': lambda x: x >= 8.0            
        }
    },
    'Giai_Tri_MXH': {
        'weights': {'perf': 0.2, 'cam': 0.1, 'batt': 0.3, 'disp': 0.4},
        'hard_filters': {'display_tier': lambda x: x >= 3}
    },
    'Chup_Hinh': {
        'weights': {'perf': 0.15, 'cam': 0.6, 'batt': 0.1, 'disp': 0.15},
        'hard_filters': {
            'score_cam': lambda x: x >= CAM_THRESHOLD,
            'storage_gb': lambda x: x >= 128.0,
            'rear_main_camera_mp': lambda x: x >= 50.0,  # Đảm bảo camera chính độ phân giải cao
            'front_camera_mp': lambda x: x >= 12.0       # Tùy chọn: Đảm bảo selfie sắc nét
        }
    },
    'Quay_Phim': {
        'weights': {'perf': 0.15, 'cam': 0.6, 'batt': 0.1, 'disp': 0.15},
        'hard_filters': {
            'score_cam': lambda x: x >= CAM_THRESHOLD,
            'storage_gb': lambda x: x >= 256.0,
            'video_resolution_rank': lambda x: x >= 4, #4=4K
            'fps_at_max_resolution': lambda x: x >= 30,
        }
    },
    'Pin_Trau_Sac_Nhanh': {
        'weights': {'perf': 0.1, 'cam': 0.1, 'batt': 0.7, 'disp': 0.1},
        'hard_filters': {
            'battery_mah': lambda x: x >= 6000,
            'fast_charge_w': lambda x: x >= 33,
        }
    },

    'Nho_Gon_Nhe_Tay': {
        'weights': {'perf': 0.2, 'cam': 0.3, 'batt': 0.2, 'disp': 0.3},
        # ĐIỀU KIỆN CỨNG: Bắt buộc trọng lượng phải thuộc nhóm 1 (Dưới 185g)
        'hard_filters': {'weight_tier': lambda x: x == 1} 
    },

    'Khang_Nuoc': {
        'weights': {'perf': 0.25, 'cam': 0.25, 'batt': 0.25, 'disp': 0.25},
        'hard_filters': {'ip_status': lambda x: x == 1}
    },

    'Bat_Ky': {
        'weights': {'perf': 0.25, 'cam': 0.25, 'batt': 0.25, 'disp': 0.25},
        'hard_filters': {}
    }
}


In [98]:
# ==========================================
#  LÕI HỆ THỐNG KBRS (ENGINE)
# ==========================================
class KBRSEngine:
    def __init__(self, df_phones, mapping_matrix):
        self.df = df_phones.copy()
        self.mapping_matrix = mapping_matrix
        # Các cột điểm đã chuẩn hóa (0-10) dùng để nhân ma trận
        self.score_features = ['score_perf', 'score_cam', 'score_batt', 'score_disp']

    def recommend(self, user_needs, max_budget=None, top_k=3):
        """
        user_needs: Có thể truyền vào 1 chuỗi (String) hoặc 1 danh sách (List of Strings)
        """
        # 0. Chuẩn hóa đầu vào thành List để dễ xử lý
        if isinstance(user_needs, str):
            user_needs = [user_needs]
            
        # Lọc ra các nhu cầu hợp lệ có trong hệ thống
        valid_needs = [n for n in user_needs if n in self.mapping_matrix]
        if not valid_needs:
            return "Lỗi: Không tìm thấy nhu cầu hợp lệ nào."

        # BƯỚC 1: TÍNH TRUNG BÌNH TRỌNG SỐ CHO ĐA NHU CẦU
        combined_weights = {'perf': 0, 'cam': 0, 'batt': 0, 'disp': 0}
        
        for need in valid_needs:
            weights = self.mapping_matrix[need]['weights']
            for key in combined_weights:
                combined_weights[key] += weights.get(key, 0)
                
        # Chia trung bình để tổng trọng số không bị phình to
        num_needs = len(valid_needs)
        user_vector = np.array([
            combined_weights['perf'] / num_needs,
            combined_weights['cam'] / num_needs,
            combined_weights['batt'] / num_needs,
            combined_weights['disp'] / num_needs
        ])

        filtered_df = self.df.copy()
        # BƯỚC 2: GỘP LỌC (ÁP DỤNG SOFT CONSTRAINT CHO BUDGET)
        if max_budget:
            # Ngưỡng ngân sách mềm (Cho phép vượt tối đa 10%)
            soft_budget_limit = max_budget * 1.10
            filtered_df = filtered_df[filtered_df['price_vnd'] <= soft_budget_limit]

        # Quét qua từng nhu cầu, máy nào không thỏa mãn luật cứng sẽ bị rớt đài dần
        for need in valid_needs:
            hard_filters = self.mapping_matrix[need].get('hard_filters', {})
            for col, condition in hard_filters.items():
                if col in filtered_df.columns:
                    filtered_df = filtered_df[condition(filtered_df[col])]

        # Nếu lọc cứng chọi nhau (Ví dụ: Vừa đòi Flagship vừa đòi Giá rẻ)
        if filtered_df.empty:
            return "Không có chiếc điện thoại nào thỏa mãn ĐỒNG THỜI tất cả các nhu cầu bắt buộc của bạn."

        # BƯỚC 3: CHẤM ĐIỂM BẰNG MA TRẬN
        item_matrix = filtered_df[self.score_features].values
        # Tính điểm cơ bản từ nhu cầu người dùng
        base_scores = np.dot(item_matrix, user_vector)

        # BƯỚC 4: TÍNH TOÁN PENALTY CHO CÁC SẢN PHẨM VƯỢT NGÂN SÁCH
        if max_budget:
            penalties = []
            for price in filtered_df['price_vnd']:
                if price > max_budget:
                    # Tỷ lệ vượt quá ngân sách (từ 0% đến 10%)
                    over_ratio = (price - max_budget) / max_budget
                    # Phạt điểm: vượt tối đa 10% ngân sách sẽ bị trừ tối đa 5% điểm
                    penalty = base_scores[len(penalties)] * (over_ratio * 0.5)
                    penalties.append(penalty)
                else:
                    penalties.append(0.0)
            
            filtered_df['total_score'] = base_scores - np.array(penalties)
        else:
            filtered_df['total_score'] = base_scores

        # Xếp hạng
        final_result = filtered_df.sort_values(by='total_score', ascending=False).head(top_k)
        
        display_cols = ['product_name', 'price_vnd'] + self.score_features + ['total_score']
        return final_result[display_cols].round(2)

# Test

In [99]:
engine = KBRSEngine(df_ready_for_kbrs, MAPPING_MATRIX_CONFIG)

In [102]:
# ==========================================
# 4. CHẠY TEST CÁC KỊCH BẢN THỰC TẾ

print("==== KỊCH BẢN 1: KHÁCH HÀNG NHU CẦU CHỤP HÌNH, NGÂN SÁCH DƯỚI 15 TRIỆU ====")
display(engine.recommend(user_needs='Chup_Hinh', max_budget=15000000))
print("\n")

print("==== KỊCH BẢN 2: KHÁCH HÀNG MUA MÁY CHO PHỤ HUYNH (LIÊN LẠC CƠ BẢN) ====")
# Không có điều kiện cứng, ưu tiên pin (0.7), các điểm khác thấp
display(engine.recommend(user_needs='Lien_Lac_Co_Ban', max_budget=4000000))
print("\n")

print("==== KỊCH BẢN 3: TÌM FLAGSHIP TỐT NHẤT ====")
# Ép price_segment == 'Flagship', trọng số cân bằng
display(engine.recommend(user_needs='Flagship'))

==== KỊCH BẢN 1: KHÁCH HÀNG NHU CẦU CHỤP HÌNH, NGÂN SÁCH DƯỚI 15 TRIỆU ====


,product_name,price_vnd,score_perf,score_cam,score_batt,score_disp,total_score
69,Xiaomi 15T Pro 5G 12GB 512GB,15990000,7.17,8.46,6.88,6.29,7.53
313,Xiaomi POCO F8 Pro 5G 12GB 256GB,13990000,7.07,7.80,7.78,6.29,7.46
224,Xiaomi 15T 5G 12GB 512GB,12490000,5.98,8.29,5.84,6.29,7.40




==== KỊCH BẢN 2: KHÁCH HÀNG MUA MÁY CHO PHỤ HUYNH (LIÊN LẠC CƠ BẢN) ====


,product_name,price_vnd,score_perf,score_cam,score_batt,score_disp,total_score
348,TECNO POVA 6 NEO 8GB 128GB,3890000,2.99,7.56,5.78,2.29,5.33
78,TECNO Spark 40 8GB 256GB,3590000,2.81,7.56,4.65,2.29,4.52
173,TECNO SPARK 30 Pro 8GB 256GB Transformer,3990000,3.12,7.56,3.98,6.29,4.48




==== KỊCH BẢN 3: TÌM FLAGSHIP TỐT NHẤT ====


,product_name,price_vnd,score_perf,score_cam,score_batt,score_disp,total_score
164,OPPO Find N6 16GB 512GB,63990000,8.49,9.84,6.74,8.29,8.66
9,OPPO Find N5 16GB 512GB,44180000,8.18,9.84,6.49,8.29,8.54
338,Xiaomi 17 Ultra 5G 16GB 1TB,34440000,8.99,9.84,7.19,6.29,8.25


In [101]:
print("==== TÌM MÁY VỪA CHƠI GAME VỪA PIN TRÂU ====")
# Hệ thống sẽ yêu cầu máy đó phải có: is_heavy_gaming == True VÀ battery_beast == True
display(engine.recommend(user_needs=['Choi_Game', 'Pin_Trau_Sac_Nhanh']))

print("==== MÁY NHẸ, FLAGSHIP ====")
display(engine.recommend(user_needs=['Nho_Gon_Nhe_Tay','Flagship']))


print("\n==== TÌM MÁY CHƠI GAME NHƯNG CHỤP HÌNH ĐẸP (NGÂN SÁCH 36TR) ====")
display(engine.recommend(user_needs=['Choi_Game', 'Chup_Hinh'], max_budget=36000000))


print("\n==== TEST LỖI XUNG ĐỘT LOGIC CỦA NGƯỜI DÙNG ====")
# Khách hàng đòi Flagship cao cấp nhưng chọn nhu cầu Giá Rẻ
display(engine.recommend(user_needs=['Flagship', 'Gia_Re'])) 
# Kết quả sẽ in ra: "Không có chiếc điện thoại nào thỏa mãn ĐỒNG THỜI..."

==== TÌM MÁY VỪA CHƠI GAME VỪA PIN TRÂU ====


,product_name,price_vnd,score_perf,score_cam,score_batt,score_disp,total_score
338,Xiaomi 17 Ultra 5G 16GB 1TB,34440000,8.99,9.84,7.19,6.29,8.02
252,OPPO Find X9 Pro 16GB 512GB,31990000,8.44,9.84,7.68,6.29,8.01
186,OPPO Find X9 16GB 512GB,25990000,8.44,9.84,7.38,6.29,7.88


==== MÁY NHẸ, FLAGSHIP ====


,product_name,price_vnd,score_perf,score_cam,score_batt,score_disp,total_score
373,iPhone Air 1TB | Chính hãng,33990000,7.65,7.8,2.35,6.29,6.49
178,iPhone Air 512GB | Chính hãng,28990000,7.15,7.8,2.35,6.29,6.37
240,iPhone Air 256GB | Chính hãng,22890000,6.90,7.8,2.35,6.29,6.30



==== TÌM MÁY CHƠI GAME NHƯNG CHỤP HÌNH ĐẸP (NGÂN SÁCH 36TR) ====


,product_name,price_vnd,score_perf,score_cam,score_batt,score_disp,total_score
338,Xiaomi 17 Ultra 5G 16GB 1TB,34440000,8.99,9.84,7.19,6.29,8.70
252,OPPO Find X9 Pro 16GB 512GB,31990000,8.44,9.84,7.68,6.29,8.53
186,OPPO Find X9 16GB 512GB,25990000,8.44,9.84,7.38,6.29,8.49



==== TEST LỖI XUNG ĐỘT LOGIC CỦA NGƯỜI DÙNG ====


'Không có chiếc điện thoại nào thỏa mãn ĐỒNG THỜI tất cả các nhu cầu bắt buộc của bạn.'